# LLM graphs with loops

## Introduction

This document has simple examples of LLM-graphs with cycles (or loops).

---

## Setup

In [111]:
use LLM::Functions;
use LLM::Graph;

use Markdown::Grammar;

In [2]:
my $conf = llm-configuration('chatgpt', model => 'gpt-5.6-luna', :8192max-tokens);

LLM::Configuration(:name("chatgpt"), :model("gpt-5.6-luna"), :module("WWW::OpenAI"), :max-tokens(8192))

In [6]:
%*ENV.sort(*.key).grep(*.key.contains('_KEY')).elems

6

---

## Simplest, one-node loop (deteministic)

- Repetition is the simplest loop - it's achieved by self-reference.

- Once node state initialized, such a loop can be conceptually unrolled into a bounded sequence of passes.

```mermaid
flowchart LR
    B0 --> B1 --> B2 --> Next["..."]
```

- Initialization with "initial-value" element of the graph node spec.

- The LLM-graphs can be seen "just" as "executable graphs" (without LLM involvement)


Greate a deteministic graph with one node that has a loop:

In [17]:
my %rules =
        refine => {
            eval-function => sub ($refine) { $refine + 1 },
            initial-value => sub ($seed) { $seed },
            max-iterations => 3
        };

my $g = LLM::Graph.new(%rules, :!async);

LLM::Graph(size => 1, nodes => refine)

Plot the graph:

In [37]:
#% html
$g.create-graph.dot(:svg, theme => 'ortho', vertex-width => 0.6);

<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 14.1.2 (0)
 -->
<!-- Pages: 1 -->
 
 
 
<!-- refine -->
 
 refine 
 
 refine 
 
<!-- refine->refine -->
 
 refine->refine

Evaluate the graph and show evaluation info:

In [45]:
$g.eval(seed => 10).nodes<refine>;
.say for |$g.nodes<refine>.grep({ $_.value ~~ Numeric:D | Str:D }).sort

iteration-count => 3
max-iterations => 3
result => 13
termination-reason => max-iterations


Evaluate the graph again -- the value given to "refine" overrides the value given to to "seed":

In [25]:
say $g.eval(seed => 10, refine => 20).nodes<refine><result>;

23


All info for the node "refine":

In [26]:
.say for |$g.nodes<refine>.sort

eval-function => sub { }
initial-value => sub { }
initial-value-input => [seed]
input => [refine]
iteration-count => 3
max-iterations => 3
result => 23
spec-type => (Callable)
termination-function-input => []
termination-reason => max-iterations
test-function-input => []


---

## Simplest, one-node loop with max-iterations (LLM)

In [146]:
my $refined = llm-graph({
    draft => sub ($topic, $field) {
        "Write a few sentence explanation of $topic for first-year undergraduate $field students." 
    },
    refine => {           
        eval-function => sub ($refine) { 
            llm-synthesize(
                "Improve the following explanation to be clearer, more precise, and more concise:\n\n$refine",
                e => $conf
            )
        },
        initial-value => sub ($draft) { $draft },
        max-iterations => 3
    }
},
e => $conf
);

LLM::Graph(size => 2, nodes => draft, refine)

In [147]:
#% html
$refined.create-graph;
$refined.dot(:svg, theme => 'ortho', vertex-width => 1.2)

<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 14.1.2 (0)
 -->
<!-- Pages: 1 -->
 
 
 
<!-- draft -->
 
 draft 
 
 
 draft 
 
<!-- refine -->
 
 refine 
 
 refine 
 
<!-- draft->refine -->
 
 draft->refine 
 
 
 4 
 
<!-- refine->refine -->
 
 refine->refine

In [148]:
$refined.eval({ topic => "quantile regression", field => "mathematical fitting of bassis functions"})

LLM::Graph(size => 2, nodes => draft, refine)

Display the result:

In [149]:
#% markdown
$refined.nodes<refine><result>
.subst('\(', '$', :g)
.subst('\)', '$', :g)
.subst('\[', '$$', :g)
.subst('\]', '$$', :g)

Quantile regression models a specified conditional quantile of a response $Y$ given predictors $x$, rather than its conditional mean. For example, median regression models the conditional 50th percentile, while 90th-quantile regression estimates
$$
q_{0.9}(x)=Q_{0.9}(Y\mid x),
$$
the value below which $Y$ falls with conditional probability approximately $0.9$.

Using basis functions $\{\phi_j(x)\}_{j=1}^p$, the conditional $\tau$-quantile is modeled as
$$
q_\tau(x)=Q_\tau(Y\mid x)
=\sum_{j=1}^p \beta_j^{(\tau)}\phi_j(x),
\qquad \tau\in(0,1).
$$
The coefficients are estimated by minimizing the empirical pinball (or check) loss:
$$
\min_{\beta}\sum_{i=1}^n \rho_\tau\!\left(y_i-q_\tau(x_i)\right),
$$
where
$$
\rho_\tau(r)=
\begin{cases}
\tau r, & r\ge 0,\\
(\tau-1)r, & r<0.
\end{cases}
$$
Here $r=y-q_\tau(x)$ is the residual. The asymmetric weighting of positive and negative residuals makes the minimizer target the conditional $\tau$-quantile: for $\tau=0.5$, the loss reduces to absolute error and yields median regression, while other values of $\tau$ emphasize under- or overprediction differently.

Display node evaluation info:

In [150]:
.say for |$refined.nodes<refine>.grep(*.key ∈ <iteration-count termination-reason>)

termination-reason => max-iterations
iteration-count => 3


---

## One-node loop with max-iterations and termination function

In [ ]:
use JSON::Fast;

my $retry = llm-graph({
    json-record => {           
        eval-function => sub ($json-record) {
            # say "eval-function: ", (:$json-record);
            llm-synthesize(
                "Extract the person's name and birth year from this text as a JSON object with keys \"name\" and \"year\". Reply with the JSON only.\n\n$json-record",
                e => $conf
            )
        },
        initial-value => sub ($text) { $text },
        termination-function => sub ($json-record) {
            # say "termination-function: ", (:$json-record);
            my $h = Nil;
            try { $h = from-json($json-record) } 
            $! ?? False !! $h ~~ Map:D #&& $h.keys.sort eq <name year>
        },
        max-iterations => 3
    },
    parsed => { eval-function => sub ($json-record) { from-json($json-record) } }
},
e => $conf
);

LLM::Graph(size => 2, nodes => json-record, parsed)

In [97]:
#% html
$retry.create-graph;
$retry.dot(:svg, theme => 'ortho', vertex-width => 1.2)

<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 14.1.2 (0)
 -->
<!-- Pages: 1 -->
 
 
 
<!-- json-record -->
 
 json-record 
 
 json-record 
 
<!-- json-record->json-record -->
 
 json-record->json-record 
 
 
 
<!-- parsed -->
 
 parsed 
 
 parsed 
 
<!-- json-record->parsed -->
 
 json-record->parsed

In [ ]:
$retry
.eval({text => 'Dimitar Savatinov is born in 1980 in Bulgaria.'})
.nodes<parsed><result>

{name => Dimitar Savatinov, year => 1980}

Here is the info for the evaluation of loop node:

In [104]:
.say for |$retry.nodes<json-record>.sort

eval-function => sub { }
initial-value => sub { }
initial-value-input => [text]
input => [json-record]
iteration-count => 1
max-iterations => 3
result => {"name":"Dimitar Savatinov","year":1980}
spec-type => (Callable)
termination-function => sub { }
termination-function-input => [json-record]
termination-reason => termination-function
test-function-input => []


---

## Actor-Critic loop

- With multiple nodes loop, a starting point is needed.


- It can be "unrolled" once properly initialized:


In [0]:
flowchart LR
    B0 --> C1 --> A2 --> B1 --> Next["..."]

- A loop is anchored when one node in the cycle provides the initial state and therefore determines where the first pass begins.


- The starting point (i.e. the loop anchor) is designated implicitly by providing initialization or other loop controls in its specifications.


In [175]:
my $actorCritic = llm-graph({
	draft => sub ($idea) { "Write a first draft of a short product description for $idea ."},
    critique => sub ($refine) { 
        "Review this draft and list concrete problems:\n\n$refine\n\n" ~ 
        "If it needs no further work, reply with exactly APPROVED."
    },
    refine => {
      	eval-function => sub ($refine, $critique) {
			say (:$refine);
			say (:$critique);
			say '-' x 50;
			llm-synthesize("Improve this draft:\n\n$refine\n\nusing this critique:\n\n\$critique", e => $conf) 
		},
      	initial-value => sub ($draft) { $draft },
		termination-function => sub ($critique) { $critique ~~ Str:D && $critique.contains('APPROVED') },
      	max-iterations => 4
    }
});

LLM::Graph(size => 3, nodes => critique, draft, refine)

In [167]:
#% html
$actorCritic.create-graph;
$actorCritic.dot(:svg, theme => 'ortho', vertex-width => 1.2)

<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 14.1.2 (0)
 -->
<!-- Pages: 1 -->
 
 
 
<!-- critique -->
 
 critique 
 
 
 critique 
 
<!-- refine -->
 
 refine 
 
 refine 
 
<!-- critique->refine -->
 
 critique->refine 
 
 
 9 
 
<!-- draft -->
 
 draft 
 
 
 draft 
 
<!-- draft->refine -->
 
 draft->refine 
 
 
 4 
 
<!-- refine->critique -->
 
 refine->critique 
 
 
 
<!-- refine->refine -->
 
 refine->refine

In [176]:
$actorCritic.eval({idea => 'Wolfram Language LLM prompt that knows the schedule of a conference.'})

refine => Make your conference schedule conversational. This Wolfram Language–powered LLM prompt helps attendees find sessions, check times and locations, and plan their day using natural-language questions. Ask “What’s next?” or “Which talks cover machine learning?” to get answers based on the conference agenda.
critique => - **The product is unclear.** “Wolfram Language–powered LLM prompt” doesn’t tell attendees whether they’re using a chatbot, copying a prompt, or opening a tool. Name the format and explain how to use it.
- **“What’s next?” implies time awareness.** Clarify whether the tool knows the current conference day and local time or attendees need to provide them.
- **The agenda source is unspecified.** “Based on the conference agenda” leaves unclear whether the agenda is already loaded or users must supply it. State which.
--------------------------------------------------


Cannot convert from JSON, returning "asis".


refine => Turn your conference schedule into a conversation. Powered by the Wolfram Language, this LLM prompt helps attendees find sessions, check times and locations, and plan their day using natural-language questions. Ask “What’s next?” or “Which talks cover machine learning?” to get answers grounded in the conference agenda.
critique => (Any)
--------------------------------------------------
refine => Turn your conference schedule into an interactive conversation. Powered by the Wolfram Language, this LLM prompt helps attendees find sessions, check times and locations, and plan their day with natural-language questions. Ask, “What’s next?” or “Which talks cover machine learning?” to get clear, agenda-based answers in seconds.
critique => - **The offering is unclear.** “This LLM prompt” suggests reusable text, while “Powered by the Wolfram Language” suggests an integrated tool. Clarify what attendees will actually use.
- **The agenda connection is unexplained.** The copy promises s

LLM::Graph(size => 3, nodes => critique, draft, refine)

Display the result:

In [174]:
#% markdown
$actorCritic.nodes<refine><result>

**Conference Schedule Assistant**

Plan your conference day with an LLM-powered assistant built in the Wolfram Language. With a complete understanding of the event schedule, it can answer natural-language questions about sessions, speakers, times, and locations. Quickly find talks that match your interests, explore the program, resolve scheduling conflicts, and build a personalized itinerary—all through a simple conversation.

In [173]:
.say for |$actorCritic.nodes<refine>.grep(*.key ∈ <max-iterations termination-reason>).sort

max-iterations => 4
termination-reason => max-iterations
